In [1]:
import json
import pickle
import os
import numpy as np

In [3]:
data_path = "data/leandojo_benchmark_4/random"
preds_file = "logs/predict_retriever_random/predictions.pickle"
split = "val" # The split you want to examine

In [4]:
print(f"Loading predictions from {preds_file}...")
with open(preds_file, "rb") as f:
    preds = pickle.load(f)

Loading predictions from logs/predict_retriever_random/predictions.pickle...


/Users/isaiahkriegman/miniconda3/envs/ReProver/lib/python3.11/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


[2025-12-09 18:23:18,138] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to mps (auto detect)


[2025-12-09 18:23:18,515] torch.distributed.elastic.multiprocessing.redirects: [WARNING] NOTE: Redirects are currently not supported in Windows or MacOs.


In [5]:
preds_map = {
    (p["file_path"], p["full_name"], tuple(p["start"]), p["tactic_idx"]): p
    for p in preds
}
print(f"Loaded {len(preds_map)} predictions.")


Loaded 259580 predictions.


In [6]:
data_file_path = os.path.join(data_path, f"{split}.json")
print(f"Loading data from {data_file_path}...")
with open(data_file_path, "r") as f:
    data = json.load(f)


Loading data from data/leandojo_benchmark_4/random/val.json...


In [7]:
failures = []
print("Scanning for failures (Match not in Top 10)...")
for thm in data:
    for i, tactic in enumerate(thm["traced_tactics"]):
        # Construct the unique key to find the corresponding prediction
        key = (thm["file_path"], thm["full_name"], tuple(thm["start"]), i)
        
        if key not in preds_map:
            continue
            
        pred = preds_map[key]
        all_pos_premises = set(pred["all_pos_premises"])
        
        if len(all_pos_premises) == 0:
            continue
        retrieved_premises = pred["retrieved_premises"]
        
        # Check Top 10
        top10 = retrieved_premises[:10]
        # Failure condition: Intersection of Ground Truth and Top 10 is empty
        tp10_count = len(all_pos_premises.intersection(top10))
        
        if tp10_count == 0:
            failures.append({
                "theorem": thm["full_name"],
                "file": thm["file_path"],
                "state_before": tactic.get("state_before", "N/A"),
                "tactic_command": tactic.get("tactic", "N/A"),
                "ground_truth": list(all_pos_premises),
                "top_retrieved": top10
            })
print(f"Found {len(failures)} failure examples.")


Scanning for failures (Match not in Top 10)...
Found 2271 failure examples.


In [18]:
for idx, fail in enumerate(failures[:5]):
    print("-" * 80)
    print(f"Example {idx + 1}")
    print(f"File: {fail['file']}")
    print(f"Theorem: {fail['theorem']}")
    print(f"\nState:\n{fail['state_before']}")
    print(f"\nGround Truth Premise(s):")
    for premise in fail['ground_truth']:
        print(f"\n - {premise}")
    print(f"\nTop 3 Predictions (Incorrect):")
    for premise in fail['top_retrieved'][:3]:
        print(f"\n - {premise}")

--------------------------------------------------------------------------------
Example 1
File: Mathlib/Analysis/NormedSpace/ProdLp.lean
Theorem: WithLp.prod_nnnorm_eq_add

State:
case a
p : ℝ≥0∞
𝕜 : Type u_1
α : Type u_2
β : Type u_3
hp✝ : Fact (1 ≤ p)
inst✝¹ : SeminormedAddCommGroup α
inst✝ : SeminormedAddCommGroup β
hp : p ≠ ⊤
f : WithLp p (α × β)
⊢ ↑‖f‖₊ = ↑((‖f.1‖₊ ^ p.toReal + ‖f.2‖₊ ^ p.toReal) ^ (1 / p.toReal))

Ground Truth Premise(s):

 - Premise(path='Mathlib/Analysis/NormedSpace/ProdLp.lean', full_name='WithLp.prod_norm_eq_add', code='theorem prod_norm_eq_add (hp : 0 < p.toReal) (f : WithLp p (α × β)) :\n    ‖f‖ = (‖f.fst‖ ^ p.toReal + ‖f.snd‖ ^ p.toReal) ^ (1 / p.toReal)')

Top 3 Predictions (Incorrect):

 - Premise(path='Mathlib/Data/ENNReal/Basic.lean', full_name='ENNReal.coe_mul', code='@[simp, norm_cast] lemma coe_mul (x y : ℝ≥0) : (↑(x * y) : ℝ≥0∞) = x * y := rfl')
--------------------------------------------------------------------------------
Example 2
File: Mathli

In [23]:
retrieved_premises

[Premise(path='Mathlib/Topology/Instances/ENNReal.lean', full_name='Real.ediam_Icc', code='@[simp]\ntheorem ediam_Icc (a b : ℝ) : EMetric.diam (Icc a b) = ENNReal.ofReal (b - a)')]